# inspecting datasets

In [1]:
from src.extract import extract, dataframes
import pandas as pd

In [2]:
for key, value in dataframes.items():
        # Investigating missing values

        # Getting datasets that has the null values
        # getting total column values for calculating total rows and columns
        column_with_nulls = value.columns[value.isna().any()].tolist()
        if column_with_nulls:
            print(key) # dataset name
            # Getting columns that has null values
            for column_name in column_with_nulls:
                # Calculating percentage for null values
                null_counts = value[column_name].isna().sum()
                total_count = value.shape[0]
                null_value_percentage = (null_counts / total_count) * 100
                print(
                    f"{column_name}: {null_counts} missing values "
                    f"({null_value_percentage:.2f}%)"
                )

            print() # for new line

olist_orders_dataset
order_approved_at: 160 missing values (0.16%)
order_delivered_carrier_date: 1783 missing values (1.79%)
order_delivered_customer_date: 2965 missing values (2.98%)

olist_order_reviews_dataset
review_comment_title: 87656 missing values (88.34%)
review_comment_message: 58247 missing values (58.70%)

olist_products_dataset
product_category_name: 610 missing values (1.85%)
product_name_lenght: 610 missing values (1.85%)
product_description_lenght: 610 missing values (1.85%)
product_photos_qty: 610 missing values (1.85%)
product_weight_g: 2 missing values (0.01%)
product_length_cm: 2 missing values (0.01%)
product_height_cm: 2 missing values (0.01%)
product_width_cm: 2 missing values (0.01%)



In [ ]:
 # Investigating orders dataset for order_delivered_customer_date, by checking orders status column and other things
orders_df = dataframes['olist_orders_dataset']

null_rows = orders_df[orders_df['order_delivered_customer_date'].isna()]

filtered_null_rows = null_rows.loc[
    :, 
        [
            
            'order_id', 
            'customer_id', 
            'order_status', 
            'order_approved_at',
            'order_purchase_timestamp', 
            'order_delivered_carrier_date', 'order_delivered_customer_date'
            
        ]
    ]

In [ ]:
# print(filtered_null_rows['order_status'].value_counts()) # check the order status wise counts

print(filtered_null_rows[filtered_null_rows['order_status'] == 'delivered'])

In [ ]:
print(filtered_null_rows[(filtered_null_rows['order_status'] == 'delivered') & (filtered_null_rows['order_delivered_carrier_date'].isna())])

In [ ]:
print(filtered_null_rows[(filtered_null_rows['order_status'] == 'delivered') & (filtered_null_rows['order_approved_at'].isna())])

# Task 06 - Investigate Product Information

In [ ]:
product_df = dataframes['olist_products_dataset']

In [ ]:
product_df

In [ ]:
filtered_products_df = product_df.loc[:, [
    'product_category_name', 
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty'
]]

In [ ]:
display(filtered_products_df.head())

In [ ]:
missing_product_category_name = filtered_products_df[filtered_products_df['product_category_name'].isna()]

In [ ]:
missing_product_category_name.isna().sum().any()

In [ ]:
missing_products = filtered_products_df[filtered_products_df['product_category_name'].isna() & filtered_products_df['product_name_lenght'].isna() & filtered_products_df['product_description_lenght'].isna() & filtered_products_df['product_photos_qty'].isna()]

In [ ]:
missing_products

In [ ]:
product_df[product_df['product_id'].isna()]

In [ ]:
missing_dimensions = product_df[
    product_df[
        [
            'product_weight_g',
            'product_length_cm',
            'product_height_cm',
            'product_width_cm'
        ]
    ].isna().any(axis=1)
]

In [ ]:
missing_dimensions

In [ ]:
reviews_df = dataframes['olist_order_reviews_dataset']

In [ ]:
reviews_df.columns.tolist()

In [ ]:
reviews_df.shape

In [ ]:
# checking duplicates
reviews_df[reviews_df.duplicated()]

In [ ]:
# Checking missing records
reviews_df['review_score'].isna().sum()

In [ ]:
# checking unique 
reviews_df['review_score'].unique()

In [ ]:
# Scorewise occurances
reviews_df['review_score'].value_counts()

In [ ]:
reviews_df['review_id'].duplicated().sum()

In [ ]:
reviews_df[['review_id', 'order_id']].isna().sum()

In [ ]:
# Missing identifiers
reviews_df[['review_id', 'order_id']].isna().sum()

In [ ]:
# Duplicate identifiers
reviews_df[['review_id', 'order_id']].duplicated().sum()

In [ ]:
# Check review_id uniqueness
reviews_df['review_id'].nunique()

In [ ]:
# Review score distribution
reviews_df['review_score'].value_counts().sort_index()

In [ ]:
duplicate_reviews = reviews_df[
    reviews_df['review_id'].duplicated(keep=False)
]

duplicate_reviews.sort_values('review_id')

In [ ]:
duplicate_reviews['review_id'].value_counts().head(30)

In [ ]:
sample_review_id = duplicate_reviews['review_id'].value_counts().index[0]

duplicate_reviews[
    duplicate_reviews['review_id'] == sample_review_id
].T

In [ ]:
duplicate_reviews.groupby('review_id')['order_id'].nunique().value_counts()

In [ ]:
duplicate_reviews.groupby('review_id')['review_score'].nunique().value_counts()

# Transformation starts

- [x] Identify timestamp columns.
- [x] Understand their current Pandas data types.
- [x] Decide which columns need conversion.
- [x] Implement the conversion.
- [x] Verify the result.

In [3]:
orders_df = dataframes['olist_orders_dataset']

In [19]:
orders_df.dtypes

order_id                                    str
customer_id                                 str
order_status                           category
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [18]:
orders_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [8]:
orders_df['order_status'].unique().tolist()

['delivered',
 'invoiced',
 'shipped',
 'processing',
 'unavailable',
 'canceled',
 'created',
 'approved']

In [9]:
orders_df['order_status'] = orders_df['order_status'].astype('category')

In [10]:
orders_df['order_purchase_timestamp'] = pd.to_datetime(orders_df['order_purchase_timestamp'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [13]:
orders_df['order_approved_at'] = pd.to_datetime(orders_df['order_approved_at'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [14]:
orders_df['order_delivered_carrier_date'] = pd.to_datetime(orders_df['order_delivered_carrier_date'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [15]:
orders_df['order_delivered_customer_date'] = pd.to_datetime(orders_df['order_delivered_customer_date'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [16]:
orders_df['order_estimated_delivery_date'] = pd.to_datetime(orders_df['order_estimated_delivery_date'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [17]:
orders_df[orders_df['order_purchase_timestamp'] == 'NaT']

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


# Task 08 - Investigate order timestamp consistency

### Part 1 — Investigate chronological consistency
- [x] Approval timestamp is earlier than purchase timestamp.
- [x] Carrier delivery timestamp is earlier than approval timestamp.
- [x] Customer delivery timestamp is earlier than carrier delivery timestamp.
- [x] Customer delivery timestamp is earlier than purchase timestamp.

In [41]:
orders_df[orders_df['order_approved_at'] < orders_df['order_purchase_timestamp']].shape

(0, 8)

In [51]:
carrier_date_issued_data = orders_df[orders_df['order_delivered_carrier_date'] < orders_df['order_approved_at']]

In [53]:
customer_date_issued_data = orders_df[orders_df['order_delivered_customer_date'] < orders_df['order_delivered_carrier_date']]

In [44]:
orders_df[orders_df['order_delivered_customer_date'] < orders_df['order_purchase_timestamp']].shape

(0, 8)

In [ ]:
orders_df.head()

### Part 2 — Investigate estimated delivery
- [x] Estimated delivery date is earlier than purchase date.
- [x] Actual customer delivery is later than the estimated delivery date.

In [45]:
orders_df[orders_df['order_estimated_delivery_date'] < orders_df['order_purchase_timestamp']].shape

(0, 8)

In [55]:
estimated_delivery_date_issued_data = orders_df[orders_df['order_estimated_delivery_date'] < orders_df['order_delivered_customer_date']]

### Part 3 — Report your findings
- [x] Number of affected records.
- [x] A few sample records (order_id and relevant timestamps).

In [47]:
orders_df[['order_id', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']]

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26
...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28
99437,63943bddc261676b46f01ca7ac2f7bd8,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02
99438,83c1379a015df1e13d02aae0204711ab,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27
99439,11c177c8e97725db2631073c19f07b62,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15


In [52]:
carrier_date_issued_data.head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
15,dcb36b511fcac050b97cd5c05de84dc3,3b6828a50ffe546942b7a473d70ac0fc,delivered,2018-06-07 19:03:12,2018-06-12 23:31:02,2018-06-11 14:54:00,2018-06-21 15:34:32,2018-07-04
64,688052146432ef8253587b930b01a06d,81e08b08e5ed4472008030d70327c71f,delivered,2018-04-22 08:48:13,2018-04-24 18:25:22,2018-04-23 19:19:14,2018-04-24 19:31:58,2018-05-15
199,58d4c4747ee059eeeb865b349b41f53a,1755fad7863475346bc6c3773fe055d3,delivered,2018-07-21 12:49:32,2018-07-26 23:31:53,2018-07-24 12:57:00,2018-07-25 23:58:19,2018-07-31
210,412fccb2b44a99b36714bca3fef8ad7b,c6865c523687cb3f235aa599afef1710,delivered,2018-07-22 22:30:05,2018-07-23 12:31:53,2018-07-23 12:24:00,2018-07-24 19:26:42,2018-07-31
415,56a4ac10a4a8f2ba7693523bb439eede,78438ba6ace7d2cb023dbbc81b083562,delivered,2018-07-22 13:04:47,2018-07-27 23:31:09,2018-07-24 14:03:00,2018-07-28 00:05:39,2018-08-06
481,32e4fa9bb468884309b58b9348de70c3,e54367d4b00c5cb76d2dfe71b9bdb89c,delivered,2018-07-04 16:49:21,2018-07-05 16:33:06,2018-07-05 14:50:00,2018-07-07 14:41:18,2018-07-23
483,4df92d82d79c3b52c7138679fa9b07fc,ba0660bf3fffe505ee892e153a2fbd49,delivered,2018-07-24 11:32:11,2018-07-29 23:30:52,2018-07-26 14:46:00,2018-07-27 18:55:57,2018-08-06
585,16e38caa92e342c7780f68832f832d4d,d29935fdaf4a76f34653da3def4f2a24,delivered,2018-05-07 01:09:09,2018-05-07 16:52:39,2018-05-07 15:09:00,2018-05-24 00:31:18,2018-06-07
615,b9afddbdcfadc9a87b41a83271c3e888,85c6af75161b8b2b1af98e82b5a6a5a5,delivered,2018-08-16 13:50:48,2018-08-16 14:05:13,2018-08-16 13:27:00,2018-08-24 14:58:37,2018-09-04
817,6051e6d3da9a50b7325cbe9c81025062,c25122206bfcb3ffa950706e1cb10f2b,delivered,2018-07-03 23:40:16,2018-07-05 16:31:26,2018-07-04 12:14:00,2018-07-05 22:52:28,2018-07-19


In [54]:
customer_date_issued_data.head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,5f50465da00b7fed5dd1239f4ecf6e2c,delivered,2017-07-20 11:20:52,2017-07-21 06:43:14,2017-07-28 16:57:58,2017-07-25 19:32:56,2017-08-14
9553,383aa8b2724fe452d9ccd9934a8c628b,b1cb2f9d7a19480f3749e248db14d58f,delivered,2017-07-02 20:58:43,2017-07-02 21:10:20,2017-07-07 17:22:41,2017-07-06 14:27:51,2017-07-21
13487,cb1134f9010d242e9515ad1c78ec0c39,2fd33ac77677bd214b1882868317eeed,delivered,2017-07-16 12:35:34,2017-07-18 06:03:50,2017-07-20 19:22:02,2017-07-19 14:13:28,2017-08-08
14474,dceb62e8fa94b46006c9554fed743df0,2721900eb4e0f1cc2c836dd7bc1b1e11,delivered,2017-07-20 20:58:05,2017-07-22 11:45:11,2017-08-01 18:23:30,2017-07-26 18:09:10,2017-08-11
19268,5f9d46795c3126674e52becb3a1a517f,79287bcaafdde5c793b996fc40bb7d9f,delivered,2017-07-18 11:48:20,2017-07-18 12:03:29,2017-07-20 23:03:42,2017-07-20 18:52:41,2017-07-31
21338,8c78d01de3a9009e23d6877a7cc9be20,6cd7106899e59a1fbd0622d5f1efedf4,delivered,2016-10-08 15:36:50,2016-10-08 18:13:44,2016-10-26 11:41:53,2016-10-25 17:51:46,2016-11-30
22520,b27af682321527a6349f1761eb3f360c,9859dd92e872dbaa60ca3cd5f0d7ad07,delivered,2017-06-14 20:17:04,2017-06-14 20:30:08,2017-06-27 14:51:54,2017-06-26 15:45:35,2017-07-14
25393,1cc3ae63caffff2d6c3ee3e78e074acf,01c843a2c0600def0b7693dba47af460,delivered,2017-08-07 21:35:22,2017-08-08 21:45:15,2017-08-10 18:28:56,2017-08-10 18:05:38,2017-08-25
25646,e37f11cae9985ca58f0b56f268720537,3947a361301f2ff0f3223159a0f2701c,delivered,2017-07-26 11:46:34,2017-07-27 10:10:16,2017-08-01 18:17:47,2017-07-31 17:49:56,2017-08-24
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,63be4feff10a0b1d85f2cfbf10df9754,delivered,2017-07-30 19:32:23,2017-07-30 19:45:09,2017-08-09 18:18:43,2017-08-01 21:13:01,2017-08-18


In [57]:
estimated_delivery_date_issued_data.shape

(7827, 8)

In [58]:
estimated_delivery_date_issued_data.head(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
20,203096f03d82e0dffbc41ebc2e2bcfb7,d2b091571da224a1b36412c18bc3bbfe,delivered,2017-09-18 14:31:30,2017-09-19 04:04:09,2017-10-06 17:50:03,2017-10-09 22:23:46,2017-09-28
25,fbf9ac61453ac646ce8ad9783d7d0af6,3a874b4d4c4b6543206ff5d89287f0c3,delivered,2018-02-20 23:46:53,2018-02-22 02:30:46,2018-02-26 22:25:22,2018-03-21 22:03:54,2018-03-12
35,8563039e855156e48fccee4d611a3196,5f16605299d698660e0606f7eae2d2f9,delivered,2018-02-17 15:59:46,2018-02-17 16:15:34,2018-02-20 23:03:56,2018-03-20 00:59:25,2018-03-20
41,6ea2f835b4556291ffdc53fa0b3b95e8,c7340080e394356141681bd4c9b8fe31,delivered,2017-11-24 21:27:48,2017-11-25 00:21:09,2017-12-13 21:14:05,2017-12-28 18:59:23,2017-12-21
57,66e4624ae69e7dc89bd50222b59f581f,684fa6da5134b9e4dab731e00011712d,delivered,2018-03-09 14:50:15,2018-03-09 15:40:39,2018-03-15 00:31:19,2018-04-03 13:28:46,2018-04-02
58,a685d016c8a26f71a0bb67821070e398,911e4c37f5cafe1604fe6767034bf1ae,delivered,2017-03-13 18:14:36,2017-03-13 18:14:36,2017-03-22 14:03:09,2017-04-06 13:37:16,2017-03-30
97,6a0a8bfbbe700284feb0845d95e0867f,68451b39b1314302c08c65a29f1140fc,delivered,2017-11-22 11:32:22,2017-11-22 11:46:50,2017-11-27 13:39:35,2017-12-28 19:43:00,2017-12-11
102,a5474c0071dd5d1074e12d417078bbd0,ef15b3240b2083e0487762ee2978d2b8,delivered,2018-07-30 22:41:44,2018-08-01 02:50:15,2018-08-02 10:35:00,2018-08-03 19:28:47,2018-08-02
110,9d531c565e28c3e0d756192f84d8731f,d4faa220408c20e53595d2950f361f3b,delivered,2017-11-28 21:00:44,2017-12-01 09:31:15,2017-12-14 20:37:43,2018-01-23 21:38:52,2017-12-22
115,8fc207e94fa91a7649c5a5dab690272a,c69f8b33e62ecb30ff78ae46d7fb9241,delivered,2017-11-26 17:49:46,2017-11-26 17:58:34,2017-11-28 23:17:09,2018-01-20 13:42:22,2017-12-19


In [59]:
carrier_date_issued_data['order_status'].value_counts()

order_status
delivered      1350
shipped           9
approved          0
canceled          0
created           0
invoiced          0
processing        0
unavailable       0
Name: count, dtype: int64

In [60]:
customer_date_issued_data['order_status'].value_counts()

order_status
delivered      23
approved        0
canceled        0
created         0
invoiced        0
processing      0
shipped         0
unavailable     0
Name: count, dtype: int64

In [62]:
estimated_delivery_date_issued_data['order_status'].value_counts()

order_status
delivered      7826
canceled          1
approved          0
created           0
invoiced          0
processing        0
shipped           0
unavailable       0
Name: count, dtype: int64

In [65]:
total_delivered_orders_count = orders_df['order_status'].value_counts().tolist()[0]

In [66]:
late_delivered_orders = estimated_delivery_date_issued_data['order_status'].value_counts().tolist()[0]

In [67]:
late_delivery_percentage = (late_delivered_orders / total_delivered_orders_count) * 100

In [69]:
round(late_delivery_percentage, 2)

8.11